~ back from a 4-lesson break covering web foundations ~

### Web Scraping with Beautiful Soup

To access content hosted on websites which do not offer APIs, web scraping is an alternative. [BeautifulSoup4](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) is a module that helps developers parse HTML and extract relevant elements, compiling them into desired data structures.

#### Ethics of Web Scraping

Web browsers work by constantly scraping the data from all websites listed on the internet. Google lost a lawsuit claim whereby displaying [Genius' lyrics](https://genius.com) directly as a search result had caused a loss of ad revenue from the otherwise website visits. In hiQ v. LinkedIn, hiQ lost after scraping LinkedIn for commerical benefit. However, the law actually favours web scraping, as long as the data is publicly available (not held behind authentication and thus privileged) and not copyrighted (most creative work are automatically copyrighted).

Many websites use CAPTCHA (human interpretation) and reCAPTCHA (mouse movement, stored cookies) to prevent code from scraping website data.

Angela recommends following with regard to web scraping:
1. Follow the Golden Rule with web scraping
2. If they make their API publicly available, use their API
3. Respect the Web Owner (take time.pause()s). Some sites have advice for bots which are scraping their website, accessible in root/robots.txt
```
User-Agent: *
Crawl-delay: 30
Disallow: /collapse?
Disallow: /context?
Disallow: /fave?
Disallow: /flag?
Disallow: /hide?
Disallow: /login
Disallow: /logout
Disallow: /r?
Disallow: /reply?
Disallow: /submitlink?
Disallow: /vote?
Disallow: /x?
```
^ Disallowed paths and max request frequency in seconds from [HN's robot.txt](https://news.ycombinator.com/robots.txt).

In [5]:
from bs4 import BeautifulSoup
# import lxml

with open('Movie Web Scraper/website.html','r') as file:
    contents = file.read()

soup = BeautifulSoup(contents, 'html.parser') # markup_str from html file, parser type (can also take "lxml")
print(soup.prettify()) # print the html with indents

# accessing specific elements
print(soup.title) # entire element
print(soup.title.name) # name of the tag
print(soup.title.string) # string inside the tag, same as get_text() or .text

print(soup.a) # first anchor tag
print(soup.li) # first list item tag

# searching through websites for all elements with a specific tag
all_anchor_tags = soup.find_all(name='a') # find all anchor tags, returns a list of elements
soup.find_all(name='p') # find all paragraph tags

for tag in all_anchor_tags: # iterate through the list
    print(tag.getText()) # get the enclosed text
    print(tag.get('href')) # get enclosed href

# searching for elements with specific attributes
heading = soup.find(name='h1', id='name') # find the first h1 tag with id 'name'
section_heading = soup.find(name='h3', class_='heading') # find the first h3 tag with class 'heading'
# 'class' is reserved in Python, so we use 'class_' instead

print(section_heading.getText())
print(section_heading.get('class')) # get the class

# logic
company_url = soup.select_one(selector='p a') # the only anchor tag that is nested inside a paragraph tag
name = soup.select_one(selector='#name') # the element with id 'name'
headings = soup.select(selector='.heading') # all elements with class 'heading'

# find_all and select are similar, but select uses CSS selectors, which can be more powerful and flexible. For example, you can use descendant selectors, child selectors, attribute selectors, etc.


<!DOCTYPE html>
<html>
 <head>
  <meta charset="utf-8"/>
  <title>
   Angela's Personal Site
  </title>
 </head>
 <body>
  <h1 id="name">
   Angela Yu
  </h1>
  <p>
   <em>
    Founder of
    <strong>
     <a href="https://www.appbrewery.co/">
      The App Brewery
     </a>
    </strong>
    .
   </em>
  </p>
  <p>
   I am an iOS and Web Developer. I love coffee and motorcycles.
  </p>
  <hr/>
  <h3 class="heading">
   Books and Teaching
  </h3>
  <ul>
   <li>
    The Complete iOS App Development Bootcamp
   </li>
   <li>
    The Complete Web Development Bootcamp
   </li>
   <li>
    100 Days of Code - The Complete Python Bootcamp
   </li>
  </ul>
  <hr/>
  <h3 class="heading">
   Other Pages
  </h3>
  <a href="https://angelabauer.github.io/cv/hobbies.html">
   My Hobbies
  </a>
  <a href="https://angelabauer.github.io/cv/contact-me.html">
   Contact Me
  </a>
 </body>
</html>

<title>Angela's Personal Site</title>
title
Angela's Personal Site
<a href="https://www.appbrewery.co/">The 

In [35]:
# scrape a snapshot of the ycombinator forum Hacker News
from bs4 import BeautifulSoup
import requests

response = requests.get('https://news.ycombinator.com/news')
yc_webpage = response.text # equivalent to file.read() for html files, returns the html as a string

soup = BeautifulSoup(yc_webpage, 'html.parser') # turn into soup object

# information we want are in titles and post points
print(soup.title.string)
# find only returns the first matching, use select or find_all
titleline_spans = soup.select(".title .titleline")
scores_spans = soup.select(".subtext .subline")

# list comprehension to extract text and href
titles = [span.select_one("a").getText() for span in titleline_spans]
links = [span.select_one("a").get("href") for span in titleline_spans]
points = [span.select_one(selector=".score").getText() for span in scores_spans]
points = [int(point.split()[0]) for point in points] # just the number

# sorting top 3 articles by points and return their titles and hrefs
top_article = ("","",0) # title, link and points

for index, point in enumerate(points):
    if point > top_article[2]:
        top_article = (titles[index], links[index], points[index])
print(top_article)

# angela's solution without iteration
largest_number = max(points)
largest_index = points.index(largest_number)
print(titles[largest_index])
print(links[largest_index])
print(points[largest_index])


Hacker News
("Canada's bill C-22 mandates mass metadata surveillance", 'https://www.michaelgeist.ca/2026/03/a-tale-of-two-bills-lawful-access-returns-with-changes-to-warrantless-access-but-dangerous-backdoor-surveillance-risks-remains/', 916)
Canada's bill C-22 mandates mass metadata surveillance
https://www.michaelgeist.ca/2026/03/a-tale-of-two-bills-lawful-access-returns-with-changes-to-warrantless-access-but-dangerous-backdoor-surveillance-risks-remains/
916


#### Day 45: Top 100 Movies Extractor

Scrape the [Top 100 Greatest Movies of All Time](https://web.archive.org/web/20200518073855/https://www.empireonline.com/movies/features/best-movies-2/) from an Empire.com compilation, creating a text file storing the titles of each movie


In [ ]:
from bs4 import BeautifulSoup
import requests
import random

URL = 'https://web.archive.org/web/20200518073855/https://www.empireonline.com/movies/features/best-movies-2/'

response = requests.get(URL)
webpage = response.text

soup = BeautifulSoup(webpage, 'html.parser')
# print(soup.prettify())

titles_list = soup.select('h3.title')
# or titles_list = soup.find_all(name='h3', class_='title')
all_titles = [title.getText() for title in titles_list]
all_titles.reverse()
# or all_titles[::-1] using the split operator
for title in all_titles:
    print(title)

with open('/Movie Web Scraper/movies.txt', 'w') as file:
    for title in all_titles:
        file.write(f"{title}\n")

watch_today = random.choice(all_titles).split(" ", 1) # no further splits after just the one
number = watch_today[0].replace(")", "")
print(f"You are watching {watch_today[1]} today, which was no. {number} on Empire.com's list of Top 100 Greatest Movies of All Time!. Enjoy!")

1) The Godfather
2) The Empire Strikes Back
3) The Dark Knight
4) The Shawshank Redemption
5) Pulp Fiction
6) Goodfellas
7) Raiders Of The Lost Ark
8) Jaws
9) Star Wars
10) The Lord Of The Rings: The Fellowship Of The Ring
11) Back To The Future
12: The Godfather Part II
13) Blade Runner
14) Alien
15) Aliens
16) The Lord Of The Rings: The Return Of The King
17) Fight Club
18) Inception
19) Jurassic Park
20) Die Hard
21) 2001: A Space Odyssey
22) Apocalypse Now
23) The Lord Of The Rings: The Two Towers
24) The Matrix
25) Terminator 2: Judgment Day
26) Heat
27) The Good, The Bad And The Ugly
28) Casablanca
29) The Big Lebowski
30) Seven
31) Taxi Driver
32) The Usual Suspects
33) Schindler's List
34) Guardians Of The Galaxy
35) The Shining
36) The Departed
37) The Thing
38) Mad Max: Fury Road
39) Saving Private Ryan
40) 12 Angry Men
41) Eternal Sunshine Of The Spotless Mind
42) There Will Be Blood
43) One Flew Over The Cuckoo's Nest
44) Gladiator
45) Drive
46) Citizen Kane
47) Interstella

### Day 46: Creating a Spotify Playlist using the Musical Time Machine

### Day 47: Automated Amazon Price Tracker

### Selenium Webdriver Browser

#### Day 48: Game Playing Bot

### Day 49: Automating your Exercise Routine at the Gym

### Day 50: Auto Tinder Swiping Bot

### Day 51: Internet Speed Twitter Complaint Bot

### Day 52: Instagram Follower Bot

### Day 53: Data Entry Job Automation (Web Scraping Capstone)

### Web Development with Flask

#### Day 54:

### HTML and URL Parsing in Flask

#### Day 55: Higher Lower Game

### Rendering HTML/Static Files and Website Templates

#### Day 56:

### Templating with Jinja in Flask Applications

#### Day 57: